# GAN Oversampling + GAN-Based Classification (Steps 17-19)

**How to use this notebook:** run it as a continuation of your existing
`UNSW-NB15_binary-classification-complete.ipynb` kernel, right after your
Step 16 (SMOTE Baseline) cells. It reuses these variables from your notebook
as-is — nothing is re-loaded or re-scaled:

- `X_train_scaled`, `y_train`, `X_val_scaled_sel`, `y_val` (Steps 11-12)
- `feature_columns`, `feature_columns_selected` (Step 14)
- `results`, `fitted_models` dicts (Step 15 onward)
- sklearn / xgboost / lightgbm imports already in your Step 1 cell

If any of these are missing, either paste these cells directly into your
existing notebook after Step 16, or re-run Steps 1-16 first in this kernel.

**Caveat on Step 19:** your current notebook trains `baseline_models` and
`advanced_models` (Step 15/16) on `X_train_resampled_sel` — i.e. **all**
existing results are already SMOTE-based, not a true original-imbalanced
baseline. The comparison below labels them accordingly rather than
mislabeling them as "Original." Add a genuine original-imbalanced baseline
first if you want a true three-way Original vs SMOTE vs GAN comparison.

## 17. GAN Oversampling

In [ ]:
# Run once per environment. Skip if torch is already installed.
! pip install torch --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### 17.1 Prepare minority data

In [ ]:
# Reconstruct the pre-SMOTE, feature-selected training set — this is the base
# the GAN trains and oversamples on (parallel to X_train_resampled_sel, but
# without SMOTE applied).
keep_idx = [feature_columns.index(c) for c in feature_columns_selected]

X_train_scaled_sel = X_train_scaled[:, keep_idx]   # original (imbalanced), scaled, selected features
y_train_arr = np.asarray(y_train)

class_counts = pd.Series(y_train_arr).value_counts()
minority_label = class_counts.idxmin()
majority_label = class_counts.idxmax()
minority_count = int(class_counts[minority_label])
majority_count = int(class_counts[majority_label])
n_to_generate = majority_count - minority_count

X_minority = X_train_scaled_sel[y_train_arr == minority_label]

print(f"Minority class: {minority_label} ({minority_count} samples)")
print(f"Majority class: {majority_label} ({majority_count} samples)")
print(f"Synthetic samples needed to balance: {n_to_generate}")

### 17.2 Train GAN (WGAN-GP)

Wasserstein GAN with gradient penalty — more stable than a vanilla GAN
(avoids the discriminator collapse seen in the from-scratch NumPy version).

In [ ]:
n_features = X_minority.shape[1]
latent_dim = 32

class Generator(nn.Module):
    def __init__(self, latent_dim, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, n_features),
        )

    def forward(self, z):
        return self.net(z)

class Critic(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x)

generator = Generator(latent_dim, n_features).to(device)
critic = Critic(n_features).to(device)

g_opt = optim.Adam(generator.parameters(), lr=1e-4, betas=(0.5, 0.9))
c_opt = optim.Adam(critic.parameters(), lr=1e-4, betas=(0.5, 0.9))

X_minority_t = torch.tensor(X_minority, dtype=torch.float32, device=device)


def gradient_penalty(critic, real, fake, device):
    batch_size = real.size(0)
    epsilon = torch.rand(batch_size, 1, device=device)
    interpolated = (epsilon * real + (1 - epsilon) * fake).requires_grad_(True)
    scores = critic(interpolated)
    grads = torch.autograd.grad(
        outputs=scores, inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True, retain_graph=True,
    )[0]
    grad_norm = grads.view(batch_size, -1).norm(2, dim=1)
    return ((grad_norm - 1) ** 2).mean()


n_epochs = 500
batch_size = min(128, X_minority_t.shape[0])
n_critic = 5
lambda_gp = 10.0

history = {"critic_loss": [], "gen_loss": []}

for epoch in range(n_epochs):
    idx = torch.randint(0, X_minority_t.shape[0], (batch_size,))
    real = X_minority_t[idx]

    for _ in range(n_critic):
        z = torch.randn(batch_size, latent_dim, device=device)
        fake = generator(z).detach()

        c_opt.zero_grad()
        gp = gradient_penalty(critic, real, fake, device)
        c_loss = critic(fake).mean() - critic(real).mean() + lambda_gp * gp
        c_loss.backward()
        c_opt.step()

    z = torch.randn(batch_size, latent_dim, device=device)
    fake = generator(z)
    g_opt.zero_grad()
    g_loss = -critic(fake).mean()
    g_loss.backward()
    g_opt.step()

    history["critic_loss"].append(c_loss.item())
    history["gen_loss"].append(g_loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{n_epochs} | critic loss: {c_loss.item():.4f} | gen loss: {g_loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(history["critic_loss"], label="Critic loss")
plt.plot(history["gen_loss"], label="Generator loss")
plt.title("WGAN-GP training curves")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

### 17.3 Generate synthetic samples

In [ ]:
generator.eval()
with torch.no_grad():
    z = torch.randn(n_to_generate, latent_dim, device=device)
    X_synthetic = generator(z).cpu().numpy()
generator.train()

y_synthetic = np.full(n_to_generate, minority_label)

print(f"Generated {X_synthetic.shape[0]} synthetic '{minority_label}'-class samples "
      f"with {X_synthetic.shape[1]} features")

### 17.4 Validate synthetic data

A PCA projection to check the synthetic points land inside the real
minority-class cloud (not scattered randomly, not collapsed to one point),
plus a feature-wise mean/std sanity check.

In [ ]:
pca = PCA(n_components=2, random_state=42)

rng = np.random.RandomState(42)
majority_idx = rng.choice(
    np.where(y_train_arr == majority_label)[0],
    size=min(2000, majority_count), replace=False,
)
X_majority_sample = X_train_scaled_sel[majority_idx]

pca_input = np.vstack([X_minority, X_synthetic, X_majority_sample])
pca_coords = pca.fit_transform(pca_input)

n_min, n_syn = len(X_minority), len(X_synthetic)

plt.figure(figsize=(7, 6))
plt.scatter(*pca_coords[:n_min].T, s=8, alpha=0.5, label=f"Real minority ({minority_label})")
plt.scatter(*pca_coords[n_min:n_min + n_syn].T, s=8, alpha=0.5, label="Synthetic (GAN)")
plt.scatter(*pca_coords[n_min + n_syn:].T, s=8, alpha=0.3, label=f"Majority sample ({majority_label})")
plt.title("PCA projection: real minority vs. synthetic vs. majority")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.show()

real_mean, real_std = X_minority.mean(axis=0), X_minority.std(axis=0)
syn_mean, syn_std = X_synthetic.mean(axis=0), X_synthetic.std(axis=0)
mean_diff = np.abs(real_mean - syn_mean).mean()
std_diff = np.abs(real_std - syn_std).mean()

print(f"Mean absolute difference in feature means (real vs synthetic): {mean_diff:.4f}")
print(f"Mean absolute difference in feature stds  (real vs synthetic): {std_diff:.4f}")
print("Smaller is better — large values suggest the GAN hasn't converged "
      "(consider more epochs or a lower critic learning rate).")

### 17.5 Create GAN-balanced dataset

In [ ]:
X_train_gan_sel = np.vstack([X_train_scaled_sel, X_synthetic])
y_train_gan = np.concatenate([y_train_arr, y_synthetic])

print("Before GAN oversampling:", pd.Series(y_train_arr).value_counts().to_dict())
print("After  GAN oversampling:", pd.Series(y_train_gan).value_counts().to_dict())

## 18. GAN-Based Classification

In [ ]:
gan_models = {
    "Random Forest (GAN)": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost (GAN)": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric="logloss", random_state=42, n_jobs=-1,
    ),
    "LightGBM (GAN)": LGBMClassifier(
        n_estimators=200, max_depth=-1, learning_rate=0.1,
        random_state=42, n_jobs=-1, verbosity=-1,
    ),
}

for name, clf in gan_models.items():
    clf.fit(X_train_gan_sel, y_train_gan)
    y_pred = clf.predict(X_val_scaled_sel)

    results[name] = {
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
    }
    fitted_models[name] = clf

pd.DataFrame(results).T.loc[list(gan_models.keys())]

In [ ]:
voting_clf_gan = VotingClassifier(
    estimators=[
        ("rf", fitted_models["Random Forest (GAN)"]),
        ("xgb", fitted_models["XGBoost (GAN)"]),
        ("lgbm", fitted_models["LightGBM (GAN)"]),
    ],
    voting="soft",
)
voting_clf_gan.fit(X_train_gan_sel, y_train_gan)
y_pred_voting_gan = voting_clf_gan.predict(X_val_scaled_sel)

results["Voting Ensemble (GAN, RF+XGB+LGBM)"] = {
    "accuracy": accuracy_score(y_val, y_pred_voting_gan),
    "precision": precision_score(y_val, y_pred_voting_gan),
    "recall": recall_score(y_val, y_pred_voting_gan),
    "f1": f1_score(y_val, y_pred_voting_gan),
}
fitted_models["Voting Ensemble (GAN, RF+XGB+LGBM)"] = voting_clf_gan

pd.DataFrame(results).T.tail(4)

## 19. Validation Comparison — SMOTE vs GAN

Labeled by what each branch actually trained on (see the caveat at the top
of this notebook — your existing Step 15/16 results are SMOTE-based, not a
true original-imbalanced baseline).

In [ ]:
strategy_groups = {
    "SMOTE": [
        "Logistic Regression", "Decision Tree", "Random Forest",
        "XGBoost", "LightGBM", "Voting Ensemble (RF+XGB+LGBM)",
    ],
    "GAN": [
        "Random Forest (GAN)", "XGBoost (GAN)", "LightGBM (GAN)",
        "Voting Ensemble (GAN, RF+XGB+LGBM)",
    ],
}

comparison_rows = []
for strategy, model_names in strategy_groups.items():
    for name in model_names:
        if name in results:
            row = {"strategy": strategy, "model": name}
            row.update(results[name])
            comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index(["strategy", "model"])
comparison_df.sort_values(["strategy", "f1"], ascending=[True, False])

In [ ]:
best_per_strategy = (
    comparison_df.reset_index()
    .sort_values("f1", ascending=False)
    .groupby("strategy")
    .first()
)
print("Best model per balancing strategy (by validation F1):")
best_per_strategy[["model", "f1", "recall", "precision", "accuracy"]]